# Can we predict next-month arrears?
Target: `arrears_next_month`. Methodology follows `checklist.md`.

**Feature decision (checklist 7).** `final_notice_sent` is recorded in the billing cycle *after* the billing month, so it is not known at prediction time (and is likely a consequence of arrears). It is dropped, as is the identifier `meter_ref`. `billing_month` is used only to order/split rows, not as a feature, because the arrears rate drifts strongly over time.

In [ ]:
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
df=pd.read_csv('data.csv',parse_dates=['billing_month']).sort_values('billing_month').reset_index(drop=True)
print(df.shape, df.meter_ref.nunique(),'accounts')
df.groupby('billing_month').arrears_next_month.mean().plot(title='Arrears rate by billing month');

## 1. Test set (checklist 1)
The model is meant for future months, so the test set is the **latest rows**: Oct–Dec 2025. Sep 2025 is dropped as a one-month gap, since a row's label refers to the following month. Accounts recur across months; with a time split the same accounts can appear on both sides, which is how the model would be used in practice (scoring existing accounts in later months).

In [ ]:
FEATS=['tariff','occupants','kwh','bill_eur','late_payments_12m']; TARGET='arrears_next_month'
test_start=pd.Timestamp('2025-10-01'); gap=pd.Timestamp('2025-09-01')
train=df[df.billing_month<gap].reset_index(drop=True)
test=df[df.billing_month>=test_start].reset_index(drop=True)
print(len(train),len(test), train[TARGET].mean(), test[TARGET].mean())
print('test accounts also in train:', test.meter_ref.isin(train.meter_ref).mean())

## 2. Cross-validation inside training rows (checklist 2)
Expanding-window folds by month: each fold trains on earlier months and validates on the next 3 months (with a 1-month gap). Test rows are never touched.

In [ ]:
months=np.sort(train.billing_month.unique())
folds=[]
for k in range(6):
    v_end=len(months)-3*(5-k); v_start=v_end-3
    tr=train.index[train.billing_month<months[v_start-1]]
    va=train.index[(train.billing_month>=months[v_start])&(train.billing_month<=months[v_end-1])]
    folds.append((tr,va))
for tr,va in folds: print(len(tr),len(va),train.loc[va,TARGET].mean().round(3))

## 3 & 4. Candidate models (named before any scoring)
1. `baseline`: predicts training prevalence (trivial).
2. `logreg`: logistic regression, C=1.
3. `logreg_C0.1`: logistic regression, stronger regularisation.
4. `hgb`: histogram gradient boosting, shallow (max_depth=3, 100 iters, lr 0.05).
5. `hgb_deep`: histogram gradient boosting (max_depth=6, 200 iters, lr 0.05).

Selection metric: mean PR-AUC across folds (positives are rare and prevalence drifts). All preprocessing (scaling, one-hot) lives in a Pipeline, so it is fit within each fold.

In [ ]:
def prep():
    return ColumnTransformer([('cat',OneHotEncoder(handle_unknown='ignore'),['tariff']),
                              ('num',StandardScaler(),['occupants','kwh','bill_eur','late_payments_12m'])])
def tree_prep():
    return ColumnTransformer([('cat',OneHotEncoder(handle_unknown='ignore'),['tariff'])],remainder='passthrough')
CANDS={
 'baseline':lambda: DummyClassifier(strategy='prior'),
 'logreg':lambda: Pipeline([('p',prep()),('m',LogisticRegression(C=1,max_iter=1000))]),
 'logreg_C0.1':lambda: Pipeline([('p',prep()),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'hgb':lambda: Pipeline([('p',tree_prep()),('m',HistGradientBoostingClassifier(max_depth=3,max_iter=100,learning_rate=0.05,random_state=0))]),
 'hgb_deep':lambda: Pipeline([('p',tree_prep()),('m',HistGradientBoostingClassifier(max_depth=6,max_iter=200,learning_rate=0.05,random_state=0))]),
}
X=train[FEATS]; y=train[TARGET]
oof={}; rows=[]
for name,mk in CANDS.items():
    oof[name]=[]
    for i,(tr,va) in enumerate(folds):
        m=mk().fit(X.loc[tr],y.loc[tr]); p=m.predict_proba(X.loc[va])[:,1]
        oof[name].append((va,p))
        rows.append(dict(model=name,fold=i,roc_auc=roc_auc_score(y.loc[va],p),pr_auc=average_precision_score(y.loc[va],p)))
cv=pd.DataFrame(rows)
summary=cv.groupby('model')[['roc_auc','pr_auc']].agg(['mean','std']).sort_values(('pr_auc','mean'),ascending=False)
summary

## 5. Choose model and threshold on CV only
Model = highest mean CV PR-AUC. Threshold = the cut-off that maximises F1 on the pooled out-of-fold predictions of that model (F1 balances catching arrears against false alarms; adjust if the business cost ratio is known).

In [ ]:
best=cv.groupby('model').pr_auc.mean().idxmax()
va_all=np.concatenate([v for v,_ in oof[best]]); p_all=np.concatenate([p for _,p in oof[best]]); y_all=y.loc[va_all].values
ths=np.linspace(0.02,0.9,89)
f1s=[f1_score(y_all,p_all>=t) for t in ths]
thr=float(ths[int(np.argmax(f1s))])
print('chosen model:',best,'| threshold:',round(thr,2),'| CV F1:',round(max(f1s),3))
print('CV baseline PR-AUC:',cv[cv.model=='baseline'].pr_auc.mean().round(3))

### Recorded choice (before test scoring)
The model and threshold printed above are final. Only `FEATS` are used; no test-set information influenced any choice.

## 6. Refit on all training rows; score the test set once

In [ ]:
final=CANDS[best]().fit(X,y)
pt=final.predict_proba(test[FEATS])[:,1]; yt=test[TARGET]
pred=pt>=thr
res=pd.DataFrame({'chosen':[roc_auc_score(yt,pt),average_precision_score(yt,pt),precision_score(yt,pred),recall_score(yt,pred),f1_score(yt,pred)],
                  'baseline':[0.5,yt.mean(),np.nan,np.nan,np.nan]},
                 index=['ROC-AUC','PR-AUC','precision@thr','recall@thr','F1@thr'])
print('test prevalence',round(yt.mean(),3),'| train prevalence',round(y.mean(),3),'| mean predicted',round(pt.mean(),3))
res.round(3)

## Conclusion
The headline is the single test-set score in section 6. Caveat: the arrears rate rises steeply through 2025 (about 3% in 2024 to about 25% by late 2025), so the test period has a much higher base rate than training. Ranking quality (AUC) should transfer better than calibrated probabilities or a fixed threshold, so threshold-based precision/recall may shift. Monitor and recalibrate as prevalence changes.